# Madewhere application
## Revisited
Sumit Dutta

12/19/2021

I would like to make the product first and continually improve the model.

The product is probably a Chrome extension that users can install. When the user browses Amazon.com the extension will tell them likely where the product is made.

Python does not work in Chrome extensions. My preferred solution as of now is to run a daemon on my server that also is supposed to serve Wildlife Watch and Bikewise. My new daemon would just be a backend to my new Chrome extension, taking in partly cleaned review text and returning the country name and related metrics. Less desirable alternatives are: that I need some Node.js code or similar code to fetch the review data.

References:
- https://towardsdatascience.com/how-to-track-sentiment-on-reddit-with-python-and-a-chrome-extension-a623d63e3a1d
- https://github.com/scrapehero-code/amazon-review-scraper

The reference article at the following link shows how to use Spacy.
* https://towardsdatascience.com/named-entity-recognition-with-nltk-and-spacy-8c4a7d88e7da

In [1]:
import numpy as np
import pandas as pd

In [2]:
import spacy
from spacy import displacy
from collections import Counter
import pprint

In [3]:
nlp = spacy.load('en_core_web_sm')

In [4]:
example_sentence = "This product is made in the USA."

In [5]:
doc = nlp(example_sentence)

In [6]:
doc

This product is made in the USA.

In [7]:
pprint.pprint([(x.text, x.label_) for x in doc.ents])

[('USA', 'GPE')]


In [8]:
pprint.pprint([(x, x.ent_iob_, x.ent_type_) for x in doc])

[(This, 'O', ''),
 (product, 'O', ''),
 (is, 'O', ''),
 (made, 'O', ''),
 (in, 'O', ''),
 (the, 'O', ''),
 (USA, 'B', 'GPE'),
 (., 'O', '')]


In [9]:
displacy.render(doc, style="dep", jupyter=True, options={"distance": 100})

In [10]:
[(x.orth_,x.pos_, x.lemma_) for x in [y for y in doc]]  # if not y.is_stop and y.pos_ != 'PUNCT']]

[('This', 'DET', 'this'),
 ('product', 'NOUN', 'product'),
 ('is', 'AUX', 'be'),
 ('made', 'VERB', 'make'),
 ('in', 'ADP', 'in'),
 ('the', 'DET', 'the'),
 ('USA', 'PROPN', 'USA'),
 ('.', 'PUNCT', '.')]

In [11]:
dict([(str(x), x.label_) for x in doc.ents])

{'USA': 'GPE'}

In [25]:
displacy.render(doc, style="ent")

Here are some instructions on how I can train a model of my own with Spacy.
* https://towardsdatascience.com/train-ner-with-custom-training-data-using-spacy-525ce748fab7
* https://www.machinelearningplus.com/nlp/training-custom-ner-model-in-spacy/

Here are other sites to reference:

Datasets
* https://www.kaggle.com

Scraping
* https://stackoverflow.com/questions/63181993/scraping-amazon-reviews-using-beautiful-soup
* https://docs.scrapy.org/en/latest/intro/overview.html

### Attempt scraping

References on learning Scrapy:
* https://stackoverflow.com/questions/63181993/scraping-amazon-reviews-using-beautiful-soup
* https://docs.scrapy.org/en/latest/intro/tutorial.html#our-first-spider
* https://docs.scrapy.org/en/latest/topics/selectors.html#scrapy.selector.Selector
* https://docs.scrapy.org/en/latest/topics/settings.html
* https://stackoverflow.com/questions/49193757/can-we-run-scrapy-code-outside-of-scrapy-shell
* https://stackoverflow.com/questions/40237952/get-scrapy-crawler-output-results-in-script-file-function

In [16]:
import scrapy
from scrapy import Request

class ProductReviewAmazonSpider(scrapy.Spider):
    name = 'product_review_amazon'
    start_urls = ['https://www.amazon.com/dp/B07Q6H83VY']

    def start_requests(self):
        yield Request(self.url)

    def parse(self, response):
        print("Parsing")
        print(response)
        for row in response.xpath('//div[@class="a-section review aok-relative"]'):  # response.css("div.review"): # response.css('div#cm-cr-dp-review-list'): # response.css('div.reviews-content'): # response.css("div.review-views"):
            print(row)
            item = {}

            item['author'] = row.css('span.a-profile-name::text').extract_first()

            rating = row.css('i.review-rating > span::text').extract_first().strip().split(' ')[0]
            item['rating'] = int(float(rating.strip().replace(',', '.')))

            item['title'] = row.css('span.review-title > span::text').extract_first()
            created_date = row.css('span.review-date::text').extract_first().strip()
            item['created_date'] = created_date

            review_content = row.css('div.reviewText ::text').extract()
            review_content = [rc.strip() for rc in review_content if rc.strip()]
            item['content'] = ', '.join(review_content)

            print(item)
            yield item

In [17]:
from scrapyscript import Job, Processor
from scrapy.crawler import CrawlerProcess
import json

job1 = Job(ProductReviewAmazonSpider, url='https://www.amazon.com/dp/B07Q6H83VY')

settings = {
    'USER_AGENT': 'Mozilla/4.0 (compatible; MSIE 7.0; Windows NT 5.1)',
    "COOKIES_ENABLED": False
}

# Create a Processor, optionally passing in a Scrapy Settings object.
processor = Processor(settings=settings)

# Start the reactor, and block until all spiders complete.
data = processor.run([job1])

# Print the consolidated results
print(json.dumps(data))

# spider = ProductReviewAmazonSpider()

# process = CrawlerProcess({
#     'USER_AGENT': 'Mozilla/4.0 (compatible; MSIE 7.0; Windows NT 5.1)'
# })

# process.crawl(ProductReviewAmazonSpider)
# process.start()

2021-06-22 01:48:17 [scrapy.utils.log] INFO: Scrapy 2.5.0 started (bot: scrapybot)
2021-06-22 01:48:17 [scrapy.utils.log] INFO: Versions: lxml 4.6.3.0, libxml2 2.9.10, cssselect 1.1.0, parsel 1.6.0, w3lib 1.22.0, Twisted 21.2.0, Python 3.9.5 (default, May 18 2021, 19:34:48) - [GCC 7.3.0], pyOpenSSL 20.0.1 (OpenSSL 1.1.1k  25 Mar 2021), cryptography 3.4.7, Platform Linux-4.4.0-19041-Microsoft-x86_64-with-glibc2.27
2021-06-22 01:48:17 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.epollreactor.EPollReactor
2021-06-22 01:48:17 [scrapy.crawler] INFO: Overridden settings:
{'COOKIES_ENABLED': False,
 'USER_AGENT': 'Mozilla/4.0 (compatible; MSIE 7.0; Windows NT 5.1)'}
2021-06-22 01:48:17 [scrapy.extensions.telnet] INFO: Telnet Password: c1f486985711fb13
2021-06-22 01:48:17 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.logstats.LogStats'

Parsing
<200 https://www.amazon.com/dp/B07Q6H83VY>
<Selector xpath='//div[@class="a-section review aok-relative"]' data='<div id="RD52BG40TLIHR" data-hook="re...'>
{'author': 'Gi', 'rating': 1, 'title': None, 'created_date': 'Reviewed in the United States on September 14, 2019', 'content': "I'm a huge fan of this franchise. Own all of the games, for both PS4 and PC. Waited a very long time for this game and I'm speechless.  You can find many reviews of the gameplay and other aspects of the game, but I'll focus on my initial thoughts and will update accordingly. First and foremost, the performance on the PS4 Slim is terrible. Frames per second is unacceptable for a split screen configuration, where scrolling between screens and reviewing the map and fighting a screen full of NPCs is horrendous. Take 2 / Gearbox couldn't even get the scaling correct with the menus, loot menus, and any text (aside from subtitles) and it's similar to reading 8 pt font on a 65 inch screen. There is no verti

2021-06-22 01:48:22 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.amazon.com/dp/B07Q6H83VY>
{'author': 'Gi', 'rating': 1, 'title': None, 'created_date': 'Reviewed in the United States on September 14, 2019', 'content': "I'm a huge fan of this franchise. Own all of the games, for both PS4 and PC. Waited a very long time for this game and I'm speechless.  You can find many reviews of the gameplay and other aspects of the game, but I'll focus on my initial thoughts and will update accordingly. First and foremost, the performance on the PS4 Slim is terrible. Frames per second is unacceptable for a split screen configuration, where scrolling between screens and reviewing the map and fighting a screen full of NPCs is horrendous. Take 2 / Gearbox couldn't even get the scaling correct with the menus, loot menus, and any text (aside from subtitles) and it's similar to reading 8 pt font on a 65 inch screen. There is no vertical split screen and no other options to improve performanc

<Selector xpath='//div[@class="a-section review aok-relative"]' data='<div id="R3PIPEC18BZXU5" data-hook="r...'>
{'author': 'Brent Figiel', 'rating': 5, 'title': None, 'created_date': 'Reviewed in the United States on September 15, 2019', 'content': 'For those who don’t know, you bought a season pass with this edition that will allow you four big content drops over the next year. You got extras... just not right now.'}


2021-06-22 01:48:22 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.amazon.com/dp/B07Q6H83VY>
{'author': 'Brent Figiel', 'rating': 5, 'title': None, 'created_date': 'Reviewed in the United States on September 15, 2019', 'content': 'For those who don’t know, you bought a season pass with this edition that will allow you four big content drops over the next year. You got extras... just not right now.'}


...

<Selector xpath='//div[@class="a-section review aok-relative"]' data='<div id="RD24UZP5FY3G5" data-hook="re...'>
{'author': 'Alvin C. Colon', 'rating': 5, 'title': None, 'created_date': 'Reviewed in the United States on September 15, 2019', 'content': 'People are weird. Guy gave it one star because he said that only comes with game. Dude, we’ve known for a long time what comes with the Super deluxe. First you get a steelbook. Second you get the medium range dlc, you can see it in the pics. Thirdly you get all the DLC for the game. Gearbox stated that there will be 4 main dlcs. It’s worth the $100. Don’t let idiots tell you otherwise. Btw... game is fantastic.'}


2021-06-22 01:48:22 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.amazon.com/dp/B07Q6H83VY>
{'author': 'Alvin C. Colon', 'rating': 5, 'title': None, 'created_date': 'Reviewed in the United States on September 15, 2019', 'content': 'People are weird. Guy gave it one star because he said that only comes with game. Dude, we’ve known for a long time what comes with the Super deluxe. First you get a steelbook. Second you get the medium range dlc, you can see it in the pics. Thirdly you get all the DLC for the game. Gearbox stated that there will be 4 main dlcs. It’s worth the $100. Don’t let idiots tell you otherwise. Btw... game is fantastic.'}


<Selector xpath='//div[@class="a-section review aok-relative"]' data='<div id="R3TJJEYBA69XID" data-hook="r...'>
{'author': 'D. Vazquez', 'rating': 1, 'title': None, 'created_date': 'Reviewed in the United States on September 13, 2019', 'content': 'NO EXTRAS! OK I pre ordered the borderlands 3 super deluxe edition. All I got was the game. I’m confused. I’m sure the game itself is five star but I’m not spending $100 for a game. Not happy'}


2021-06-22 01:48:22 [scrapy.core.scraper] DEBUG: Scraped from <200 https://www.amazon.com/dp/B07Q6H83VY>
{'author': 'D. Vazquez', 'rating': 1, 'title': None, 'created_date': 'Reviewed in the United States on September 13, 2019', 'content': 'NO EXTRAS! OK I pre ordered the borderlands 3 super deluxe edition. All I got was the game. I’m confused. I’m sure the game itself is five star but I’m not spending $100 for a game. Not happy'}
2021-06-22 01:48:22 [scrapy.core.engine] INFO: Closing spider (finished)
2021-06-22 01:48:22 [scrapy.statscollectors] INFO: Dumping Scrapy stats:
{'downloader/request_bytes': 247,
 'downloader/request_count': 1,
 'downloader/request_method_count/GET': 1,
 'downloader/response_bytes': 327055,
 'downloader/response_count': 1,
 'downloader/response_status_count/200': 1,
 'elapsed_time_seconds': 5.301053,
 'finish_reason': 'finished',
 'finish_time': datetime.datetime(2021, 6, 22, 6, 48, 22, 962340),
 'httpcompression/response_bytes': 1888899,
 'httpcompression/r

[{"author": "Gi", "rating": 1, "title": null, "created_date": "Reviewed in the United States on September 14, 2019", "content": "I'm a huge fan of this franchise. Own all of the games, for both PS4 and PC. Waited a very long time for this game and I'm speechless.  You can find many reviews of the gameplay and other aspects of the game, but I'll focus on my initial thoughts and will update accordingly. First and foremost, the performance on the PS4 Slim is terrible. Frames per second is unacceptable for a split screen configuration, where scrolling between screens and reviewing the map and fighting a screen full of NPCs is horrendous. Take 2 / Gearbox couldn't even get the scaling correct with the menus, loot menus, and any text (aside from subtitles) and it's similar to reading 8 pt font on a 65 inch screen. There is no vertical split screen and no other options to improve performance. Missions are uneventful and no concise storyline that enables campaign mode truly enjoyable. In many 

In [18]:
data

[{'author': 'Gi',
  'rating': 1,
  'title': None,
  'created_date': 'Reviewed in the United States on September 14, 2019',
  'content': "I'm a huge fan of this franchise. Own all of the games, for both PS4 and PC. Waited a very long time for this game and I'm speechless.  You can find many reviews of the gameplay and other aspects of the game, but I'll focus on my initial thoughts and will update accordingly. First and foremost, the performance on the PS4 Slim is terrible. Frames per second is unacceptable for a split screen configuration, where scrolling between screens and reviewing the map and fighting a screen full of NPCs is horrendous. Take 2 / Gearbox couldn't even get the scaling correct with the menus, loot menus, and any text (aside from subtitles) and it's similar to reading 8 pt font on a 65 inch screen. There is no vertical split screen and no other options to improve performance. Missions are uneventful and no concise storyline that enables campaign mode truly enjoyable. 

In [19]:
data[-1]["content"]

'NO EXTRAS! OK I pre ordered the borderlands 3 super deluxe edition. All I got was the game. I’m confused. I’m sure the game itself is five star but I’m not spending $100 for a game. Not happy'

#### Extract entities from these actual reviews

In [26]:
for review_struct in data:
    review = review_struct["content"]
    review_doc = nlp(review)
    print("")
    print("Review:")
    print(review)
    print("Extracted entities:")
    print([(x.text, x.label_) for x in review_doc.ents])
    displacy.render(review_doc, style="ent")  # No need for displacy.serve since within a notebook environment
    
    geopolitical_entities_found = []
    geopolitical_entities_found = [x.text for x in review_doc.ents if x.label_ == "GPE"]
    geopolitical_entities_sorted = pd.Series(geopolitical_entities_found).value_counts(sort=True, ascending=False)
    
    print("Possible countries or geopolitical places or origin:")
    print(geopolitical_entities_sorted)


Review:
I'm a huge fan of this franchise. Own all of the games, for both PS4 and PC. Waited a very long time for this game and I'm speechless.  You can find many reviews of the gameplay and other aspects of the game, but I'll focus on my initial thoughts and will update accordingly. First and foremost, the performance on the PS4 Slim is terrible. Frames per second is unacceptable for a split screen configuration, where scrolling between screens and reviewing the map and fighting a screen full of NPCs is horrendous. Take 2 / Gearbox couldn't even get the scaling correct with the menus, loot menus, and any text (aside from subtitles) and it's similar to reading 8 pt font on a 65 inch screen. There is no vertical split screen and no other options to improve performance. Missions are uneventful and no concise storyline that enables campaign mode truly enjoyable. In many aspects, you'd wish this game was more linear than it is, but it's storyline isn't inspiring at all. Only after a few ho

Possible countries or geopolitical places or origin:
Series([], dtype: int64)

Review:
For those who don’t know, you bought a season pass with this edition that will allow you four big content drops over the next year. You got extras... just not right now.
Extracted entities:
[('four', 'CARDINAL'), ('the next year', 'DATE')]


<ipython-input-26-01387535e01f>:13: DeprecationWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  geopolitical_entities_sorted = pd.Series(geopolitical_entites_found).value_counts(sort=True, ascending=False)


Possible countries or geopolitical places or origin:
Series([], dtype: int64)

Review:
My wife and I have played all of the previously released games in the series for hundreds of hours.  We played them through several times and bought all of the DLCs as they came out.  They were great fun, especially when there are so few games that have great split-screen coop.  We bought a PS4 Pro and pre-ordered the Super Deluxe edition of BL3 as soon as we realized it was about to be released.  This has been a big mistake., Unlike previously released games in the series, the split-screen is horizontal only.  This may not seem like a big deal, but when combined with the tiny font, this is a nightmare on our 55".  In split screen, the center of each screen is the only thing in focus, with the images stretching on each side like you're seeing everything through a fisheye lens.  Combined with the tiny font in all of the menus, this was literally giving us headaches., Supposedly there are performance i

Possible countries or geopolitical places or origin:
Series([], dtype: int64)

Review:
I could not be more disheartened as sad about Borderlands 3. My wife and are huge, long-time fans of the series and have played all of the previous entries together all the way through. We were beyond ecstatic to get our hands on the newest installment., That is, until we actually tried playing it., The game LOOKS great initially but as many others have mentioned, the option of vertical split-screen is gone for the first time in the series. And the horizontal split-screen is nearly unplayable. You can hardly make out anything (enemies, loot, ammo types) and trying to read anything in your ECHO or compare guns is virtually impossible without standing about two feet from the TV. There is an option to lower the FOV, which does reduce the "fish-eye" issue quite a bit, but it does nothing for how small everything is when crammed into the narrow slit that the horizontal split gives you., We didn't get far 

Possible countries or geopolitical places or origin:
Series([], dtype: int64)

Review:
People are weird. Guy gave it one star because he said that only comes with game. Dude, we’ve known for a long time what comes with the Super deluxe. First you get a steelbook. Second you get the medium range dlc, you can see it in the pics. Thirdly you get all the DLC for the game. Gearbox stated that there will be 4 main dlcs. It’s worth the $100. Don’t let idiots tell you otherwise. Btw... game is fantastic.
Extracted entities:
[('Guy', 'PERSON'), ('one', 'CARDINAL'), ('First', 'ORDINAL'), ('Second', 'ORDINAL'), ('Thirdly', 'ORDINAL'), ('DLC', 'ORG'), ('Gearbox', 'PERSON'), ('4', 'CARDINAL'), ('100', 'MONEY')]


Possible countries or geopolitical places or origin:
Series([], dtype: int64)

Review:
NO EXTRAS! OK I pre ordered the borderlands 3 super deluxe edition. All I got was the game. I’m confused. I’m sure the game itself is five star but I’m not spending $100 for a game. Not happy
Extracted entities:
[('3', 'CARDINAL'), ('five', 'CARDINAL'), ('100', 'MONEY')]


Possible countries or geopolitical places or origin:
Series([], dtype: int64)
